In [11]:
import os
import re
import string
import pandas as pd

import numpy as np
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import scipy.sparse
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec

In [16]:
# ======================== Configuration ========================
CONFIG = {
    "data_path": "/Users/ashishpal/Documents/MLOPs-Vikas-Das/Movie-Review-Sentiment-Prediction/notebooks/IMDB.csv",
    "test_size": 0.2,
    "mlflow_tracking_uri": "https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow",
    "dagshub_repo_owner": "ashishpal003",
    "dagshub_repo_name": 'Movie-Review-Sentiment-Prediction',
    "experiment_name": "Bow vs TfIdf vs word2vec"
}

# ======================== SETUP MLflow & DAGSHUB ========================
mlflow.set_tracking_uri(CONFIG["mlflow_tracking_uri"])
dagshub.init(repo_owner=CONFIG["dagshub_repo_owner"], repo_name=CONFIG["dagshub_repo_name"], mlflow=True)
mlflow.set_experiment(CONFIG["experiment_name"])

Initialized MLflow to track repo "ashishpal003/Movie-Review-Sentiment-Prediction"

Repository ashishpal003/Movie-Review-Sentiment-Prediction initialized!

<Experiment: artifact_location='mlflow-artifacts:/83ce3f6a20764e35a6ca7ef5a25c88c7', creation_time=1772899166585, experiment_id='2', last_update_time=1772899166585, lifecycle_stage='active', name='Bow vs TfIdf vs word2vec', tags={}, workspace='default'>

In [5]:
# ========================== TEXT PREPROCESSING ==========================
def lemmatization(text):
    lemmatizer = WordNetLemmatizer()
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

def remove_stop_words(text):
    stop_words = set(stopwords.words("english"))
    return " ".join([word for word in text.split() if word not in stop_words])

def removing_numbers(text):
    return ''.join([char for char in text if not char.isdigit()])

def lower_case(text):
    return text.lower()

def removing_punctuations(text):
    return re.sub(f"[{re.escape(string.punctuation)}]", ' ', text)

def removing_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def normalize_text(df):
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f"Error during text normalization: {e}")
        raise

In [6]:
# ========================== LOAD & PREPROCESS DATA ==========================
def load_data(file_path):
    try:
        df = pd.read_csv(file_path)
        df = normalize_text(df)
        df = df[df['sentiment'].isin(['positive', 'negative'])]
        df['sentiment'] = df['sentiment'].replace({'negative': 0, 'positive': 1}).infer_objects(copy=False)
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        raise

In [13]:
# ========================== FEATURE ENGINEERING ==========================
VECTORIZERS = {
    'BoW': CountVectorizer(),
    'TF-IDF': TfidfVectorizer(),
    'Word2Vec': Word2Vec(min_count=1, vector_size=100, window=5)
}

ALGORITHMS = {
    'LogisticRegression': LogisticRegression(),
    'MultinomialNB': MultinomialNB(),
    'XGBoost': XGBClassifier(),
    'RandomForest': RandomForestClassifier(),
    'GradientBoosting': GradientBoostingClassifier()
}

In [14]:
# ========================== TRAIN & EVALUATE MODELS ==========================
def train_and_evaluate(df):
    with mlflow.start_run(run_name="All Experiments") as parent_run:
        for algo_name, algorithm in ALGORITHMS.items():
            for vec_name, vectorizer in VECTORIZERS.items():
                with mlflow.start_run(run_name=f"{algo_name} with {vec_name}", nested=True) as child_run:
                    try:
                        if vec_name != 'Word2Vec':
                            # Feature extraction
                            X = vectorizer.fit_transform(df['review'])
                            y = df['sentiment']
                            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=CONFIG["test_size"], random_state=42)

                            # Log preprocessing parameters
                            mlflow.log_params({
                                "vectorizer": vec_name,
                                "algorithm": algo_name,
                                "test_size": CONFIG["test_size"]
                            })

                            # Train model
                            model = algorithm
                            model.fit(X_train, y_train)

                            # Log model parameters
                            log_model_params(algo_name, model)

                            # Evaluate model
                            y_pred = model.predict(X_test)
                            metrics = {
                                "accuracy": accuracy_score(y_test, y_pred),
                                "precision": precision_score(y_test, y_pred),
                                "recall": recall_score(y_test, y_pred),
                                "f1_score": f1_score(y_test, y_pred)
                            }
                            mlflow.log_metrics(metrics)

                            # Log model
                            # mlflow.sklearn.log_model(model, "model")
                            input_example = X_test[:5] if not scipy.sparse.issparse(X_test) else X_test[:5].toarray()
                            mlflow.sklearn.log_model(model, "model", input_example=input_example)

                            # Print results for verification
                            print(f"\nAlgorithm: {algo_name}, Vectorizer: {vec_name}")
                            print(f"Metrics: {metrics}")
                        elif vec_name == 'Word2Vec':
                            data = []
                            for i in df['review']:
                                temp = []
                                for j in word_tokenize(i):
                                    temp.append(j)
                                
                                data.append(temp)
                            
                            vec_model = vectorizer.build_vocab(data, update=True)

                            def get_avg_vector(sentence, model):
                                vector_sum = np.zeros(vec_model.vector_size)
                                count = 0
                                for word in sentence:
                                    if word in model.wv:
                                        vector_sum += model.wv[word]
                                        count += 1
                                if count == 0:
                                    return vector_sum
                                return vector_sum/count
                            
                            vec_sentence = []

                            for i in data:
                                vec = get_avg_vector(i, vec_model)
                                vec_sentence.append(vec)
                            
                            X = np.array(vec_sentence)
                            y = df['sentiment']

                            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=CONFIG["test_size"], random_state=42)

                            # Log preprocessing parameters
                            mlflow.log_params({
                                "vectorizer": vec_name,
                                "algorithm": algo_name,
                                "test_size": CONFIG["test_size"]
                            })

                            # Train model
                            model = algorithm
                            model.fit(X_train, y_train)

                            # Log model parameters
                            log_model_params(algo_name, model)

                            # Evaluate model
                            y_pred = model.predict(X_test)
                            metrics = {
                                "accuracy": accuracy_score(y_test, y_pred),
                                "precision": precision_score(y_test, y_pred),
                                "recall": recall_score(y_test, y_pred),
                                "f1_score": f1_score(y_test, y_pred)
                            }
                            mlflow.log_metrics(metrics)

                            # Log model
                            # mlflow.sklearn.log_model(model, "model")
                            input_example = X_test[:5] if not scipy.sparse.issparse(X_test) else X_test[:5].toarray()
                            mlflow.sklearn.log_model(model, "model", input_example=input_example)

                            # Print results for verification
                            print(f"\nAlgorithm: {algo_name}, Vectorizer: {vec_name}")
                            print(f"Metrics: {metrics}")

                    except Exception as e:
                        print(f"Error in training {algo_name} with {vec_name}: {e}")
                        mlflow.log_param("error", str(e))

def log_model_params(algo_name, model):
    """Logs hyperparameters of the trained model to MLflow."""
    params_to_log = {}
    if algo_name == 'LogisticRegression':
        params_to_log["C"] = model.C
    elif algo_name == 'MultinomialNB':
        params_to_log["alpha"] = model.alpha
    elif algo_name == 'XGBoost':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["learning_rate"] = model.learning_rate
    elif algo_name == 'RandomForest':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["max_depth"] = model.max_depth
    elif algo_name == 'GradientBoosting':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["learning_rate"] = model.learning_rate
        params_to_log["max_depth"] = model.max_depth

    mlflow.log_params(params_to_log)


In [17]:
# ========================== EXECUTION ==========================
df = load_data(CONFIG["data_path"])
train_and_evaluate(df)

/var/folders/n8/j0_rrlzd54b4xk3yllh67k000000gn/T/ipykernel_74158/1301451151.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'negative': 0, 'positive': 1}).infer_objects(copy=False)
2026/03/07 23:22:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/07 23:22:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Algorithm: LogisticRegression, Vectorizer: BoW
Metrics: {'accuracy': 0.775, 'precision': 0.7865168539325843, 'recall': 0.7291666666666666, 'f1_score': 0.7567567567567568}
🏃 View run LogisticRegression with BoW at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2/runs/bf6f50b1c02f47ab87710f19d8db3a4e
🧪 View experiment at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2


2026/03/07 23:23:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/07 23:23:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Algorithm: LogisticRegression, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.78, 'precision': 0.7888888888888889, 'recall': 0.7395833333333334, 'f1_score': 0.7634408602150538}
🏃 View run LogisticRegression with TF-IDF at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2/runs/4bac25e65b8b48da85ff3bcdf9269e3a
🧪 View experiment at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2
Error in training LogisticRegression with Word2Vec: You cannot do an online vocabulary-update of a model which has no prior vocabulary. First build the vocabulary of your model with a corpus before doing an online update.
🏃 View run LogisticRegression with Word2Vec at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2/runs/f23dedc32ac8468db87ae08aba2935b4
🧪 View experiment at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2


2026/03/07 23:23:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/07 23:23:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...


Algorithm: RandomForest, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.775, 'precision': 0.7741935483870968, 'recall': 0.75, 'f1_score': 0.7619047619047619}
🏃 View run RandomForest with TF-IDF at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2/runs/f410e0157e404136aed92d6284cdf59c
🧪 View experiment at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2
Error in training RandomForest with Word2Vec: You cannot do an online vocabulary-update of a model which has no prior vocabulary. First build the vocabulary of your model with a corpus before doing an online update.
🏃 View run RandomForest with Word2Vec at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2/runs/b30fc701123b4d2aa8cd401b5e9c89a7
🧪 View experiment at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2


2026/03/07 23:26:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/07 23:26:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Algorithm: GradientBoosting, Vectorizer: BoW
Metrics: {'accuracy': 0.75, 'precision': 0.73, 'recall': 0.7604166666666666, 'f1_score': 0.7448979591836735}
🏃 View run GradientBoosting with BoW at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2/runs/d8c3b1e4fb5f4b03b32f9535bec4f1c8
🧪 View experiment at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2


2026/03/07 23:26:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/07 23:26:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Algorithm: GradientBoosting, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.75, 'precision': 0.7346938775510204, 'recall': 0.75, 'f1_score': 0.7422680412371134}
🏃 View run GradientBoosting with TF-IDF at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2/runs/ed3aa09d6f394da0b5810cfe2e83ab61
🧪 View experiment at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2
Error in training GradientBoosting with Word2Vec: You cannot do an online vocabulary-update of a model which has no prior vocabulary. First build the vocabulary of your model with a corpus before doing an online update.
🏃 View run GradientBoosting with Word2Vec at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2/runs/3b11dba8f6cf463fb3f00a9771ebbcaf
🧪 View experiment at: https://dagshub.com/ashishpal003/Movie-Review-Sentiment-Prediction.mlflow/#/experiments/2
🏃 View run All Experiments at: https://dagshub.com/a